# 2026 호르무즈 해협(호르무즈 해협) 영어 해외언론 텍스트마이닝 수집

**기간:** 2026-01-01 ~ 2026-08-31  
**핵심 키워드:** `"Strait of Hormuz"`  
**수집 구조:** NewsCatcher → URL 중복 제거 → Diffbot Article → Diffbot Discussion → CSV/Excel  
**주의:** API 키는 코드에 직접 입력하지 않고 환경변수로 넣습니다.

> 이 노트북은 기존 `pygooglenews + Selenium` 수집부를 API 기반으로 교체하고,
> 기존 NLTK/텍스트마이닝 분석부와 연결하기 위한 수집용 노트북입니다.


## 0. 설치

Jupyter/VS Code에서 최초 1회만 실행합니다.


In [ ]:
%pip install -q requests pandas openpyxl tqdm python-dotenv nltk scikit-learn matplotlib wordcloud

## 1. 라이브러리와 기본 설정

NewsCatcher API는 `https://v3-api.newscatcherapi.com/api/search`를 사용하고,
API 키는 `x-api-token` 헤더로 전달합니다.

Diffbot Article API는 원문 URL에서 제목, 본문, 날짜, 저자, 언론사명,
언론사 국가/지역, 언어 등을 추출할 수 있으며 댓글도 `discussion`으로 반환할 수 있습니다.


### GitHub 커밋 전 API 키 설정

이 프로젝트는 API 키를 코드에 직접 입력하지 않고 `.env` 파일에서 읽습니다.

프로젝트 루트에 `.env` 파일을 만들고 다음과 같이 입력하세요:

```text
NEWSCATCHER_API_KEY=실제_뉴스캐처_API_키
DIFFBOT_API_KEY=실제_디프봇_API_키
```

`.env`는 반드시 `.gitignore`에 등록하여 GitHub에 커밋하지 않습니다.
대신 저장소에는 `.env.example`을 두고 API 키 이름만 공유하는 것을 권장합니다.

예:

```text
NEWSCATCHER_API_KEY=
DIFFBOT_API_KEY=
```

API 키가 이미 GitHub에 커밋된 적이 있다면 `.env`로 옮기는 것만으로는 충분하지 않으므로 해당 키를 폐기하고 새 키를 발급하는 것이 안전합니다.


In [ ]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import requests
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# .env 파일은 GitHub에 커밋하지 않습니다.
# 기본적으로 현재 작업 폴더와 상위 폴더에서 .env를 찾습니다.
load_dotenv()

NEWSCATCHER_API_KEY = os.getenv("NEWSCATCHER_API_KEY")
DIFFBOT_API_KEY = os.getenv("DIFFBOT_API_KEY")

if not NEWSCATCHER_API_KEY:
    raise ValueError(
        "NEWSCATCHER_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

if not DIFFBOT_API_KEY:
    raise ValueError(
        "DIFFBOT_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

# API 키 자체는 출력하지 않습니다.
print("API 키 로드 완료")
print(f"NewsCatcher API: {'설정됨' if NEWSCATCHER_API_KEY else '없음'}")
print(f"Diffbot API: {'설정됨' if DIFFBOT_API_KEY else '없음'}")

START_DATE = "2026-01-01"
END_DATE = "2026-08-31"

QUERIES = [
    '"Strait of Hormuz"',
    '"Hormuz Strait"',
]

NEWS_URL = "https://v3-api.newscatcherapi.com/api/search"
DIFFBOT_ARTICLE_URL = "https://api.diffbot.com/v3/article"
DIFFBOT_DISCUSSION_URL = "https://api.diffbot.com/v3/discussion"

DATA_DIR = Path("hormuz_2026_data")
DATA_DIR.mkdir(exist_ok=True)

RAW_NEWS_FILE = DATA_DIR / "01_newscatcher_raw.csv"
ARTICLES_FILE = DATA_DIR / "02_articles.csv"
COMMENTS_FILE = DATA_DIR / "03_comments.csv"
FAILED_FILE = DATA_DIR / "04_failed_urls.csv"


## 2. API 연결 테스트

실제 대량 수집 전에 **각 API 키가 정상인지 먼저 확인**합니다.


In [ ]:
def test_newscatcher():
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }
    payload = {
        "q": '"Strait of Hormuz"',
        "lang": "en",
        "from_": START_DATE,
        "to_": END_DATE,
        "page_size": 1,
    }
    r = requests.post(NEWS_URL, headers=headers, json=payload, timeout=60)
    print("NewsCatcher:", r.status_code)
    r.raise_for_status()
    data = r.json()
    print("total_hits:", data.get("total_hits"))
    return data

def test_diffbot(test_url="https://www.bbc.com/news"):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": test_url,
        "discussion": "false",
    }
    r = requests.get(DIFFBOT_ARTICLE_URL, params=params, timeout=60)
    print("Diffbot:", r.status_code)
    r.raise_for_status()
    return r.json()

nc_test = test_newscatcher()


## 3. URL 정규화

검색어가 여러 개이면 같은 기사가 반복될 수 있으므로 URL을 정규화한 뒤 중복 제거합니다.


In [ ]:
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "gclid", "fbclid", "mc_cid", "mc_eid"
}

def normalize_url(url):
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()
    parts = urlsplit(url)

    if parts.scheme not in {"http", "https"}:
        return None

    query_items = []
    for item in parts.query.split("&") if parts.query else []:
        if "=" in item:
            k, v = item.split("=", 1)
        else:
            k, v = item, ""
        if k.lower() not in TRACKING_PARAMS:
            query_items.append(f"{k}={v}")

    clean = urlunsplit((
        parts.scheme.lower(),
        parts.netloc.lower(),
        parts.path.rstrip("/") or "/",
        "&".join(query_items),
        ""
    ))
    return clean


## 4. NewsCatcher 기사 목록 수집

- 영어: `lang=en`
- 기간: 2026-01-01 ~ 2026-08-31
- 페이지 크기: 최대 1000
- 검색어별 결과를 합친 뒤 URL 기준으로 중복 제거
- `exclude_duplicates=True`도 함께 사용합니다.

NewsCatcher 문서상 한 검색 요청은 최대 10,000건까지 반환되므로,
그 이상이 되는 경우에는 날짜를 월/주/일 단위로 나누어 재검색하는 것이 안전합니다.


In [ ]:
def newscatcher_search(q, from_date=START_DATE, to_date=END_DATE,
                       page=1, page_size=1000):
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }

    payload = {
        "q": q,
        "lang": "en",
        "from_": from_date,
        "to_": to_date,
        "page": page,
        "page_size": page_size,
        "exclude_duplicates": True,
        "sort_by": "date",
    }

    r = requests.post(NEWS_URL, headers=headers, json=payload, timeout=90)

    if r.status_code == 429:
        raise RuntimeError("NewsCatcher rate limit(429)입니다. 요금제 제한을 확인하고 잠시 후 재시도하세요.")

    r.raise_for_status()
    return r.json()

def collect_newscatcher(q, from_date=START_DATE, to_date=END_DATE):
    first = newscatcher_search(q, from_date, to_date, page=1, page_size=1000)

    rows = first.get("articles", [])
    total_pages = int(first.get("total_pages", 1))

    print(f"[{q}] total_hits={first.get('total_hits')} / total_pages={total_pages}")

    for page in range(2, total_pages + 1):
        data = newscatcher_search(q, from_date, to_date, page=page, page_size=1000)
        rows.extend(data.get("articles", []))
        time.sleep(0.3)

    return rows

all_raw = []

for q in QUERIES:
    try:
        all_raw.extend(collect_newscatcher(q))
    except Exception as e:
        print(f"검색 실패: {q} -> {e}")

raw_df = pd.json_normalize(all_raw)

print("수집된 raw 기사 수:", len(raw_df))
raw_df.to_csv(RAW_NEWS_FILE, index=False, encoding="utf-8-sig")

raw_df.head()


## 5. NewsCatcher 결과를 연구용 기사 목록으로 정리

API 응답 구조가 계정/버전에 따라 일부 달라질 수 있으므로,
아래 코드는 자주 사용되는 필드 후보를 순서대로 찾아 사용합니다.


In [ ]:
def first_existing(row, candidates, default=None):
    for c in candidates:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return row[c]
    return default

def make_article_index(df):
    records = []

    for _, row in df.iterrows():
        url = first_existing(row, [
            "link", "url", "clean_url", "canonical_url", "parent_url"
        ])

        if not url:
            continue

        url = normalize_url(url)
        if not url:
            continue

        records.append({
            "title_search": first_existing(row, ["title"]),
            "description_search": first_existing(row, ["summary", "excerpt", "content", "description"]),
            "source_search": first_existing(row, ["clean_url", "domain_url", "source_url", "source_name", "source"]),
            "published_search": first_existing(row, ["published_date", "published_at", "pub_date"]),
            "url": url,
            "language_search": first_existing(row, ["language", "lang"], "en"),
        })

    out = pd.DataFrame(records)
    out = out.drop_duplicates(subset=["url"]).reset_index(drop=True)

    out["article_id"] = [
        hashlib.sha1(u.encode("utf-8")).hexdigest()[:16]
        for u in out["url"]
    ]

    return out

article_index = make_article_index(raw_df)

print("URL 중복 제거 후 기사 수:", len(article_index))
article_index.head()


## 6. Diffbot Article API — 본문/언론사/국가/지역 추출

Diffbot Article API의 핵심 필드는 다음과 같습니다.

- `title`
- `text` = 전체 기사 본문
- `date`
- `author`
- `siteName`
- `publisherCountry`
- `publisherRegion`
- `humanLanguage`
- `resolvedPageUrl`
- `discussion`

기사 본문이 실제로 추출되었는지를 `body_length`와 `extraction_status`로 기록합니다.


In [ ]:
def diffbot_article(url, include_discussion=True, timeout_ms=60000):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "discussion": "true" if include_discussion else "false",
        "paging": "true",
    }

    r = requests.get(DIFFBOT_ARTICLE_URL, params=params, timeout=90)

    if r.status_code == 429:
        raise RuntimeError("Diffbot rate limit(429)입니다.")

    r.raise_for_status()
    data = r.json()

    objects = data.get("objects") or []
    if not objects:
        return None, data

    return objects[0], data

def flatten_article(article_id, source_url, obj):
    if not obj:
        return {
            "article_id": article_id,
            "source_url": source_url,
            "extraction_status": "no_object",
            "body": "",
            "body_length": 0,
        }

    body = obj.get("text") or ""

    return {
        "article_id": article_id,
        "source_url": source_url,
        "resolved_url": obj.get("resolvedPageUrl"),
        "title": obj.get("title"),
        "body": body,
        "body_length": len(body),
        "published_at": obj.get("date"),
        "estimated_date": obj.get("estimatedDate"),
        "author": obj.get("author"),
        "source": obj.get("siteName"),
        "publisher_country": obj.get("publisherCountry"),
        "publisher_region": obj.get("publisherRegion"),
        "language": obj.get("humanLanguage"),
        "num_pages": obj.get("numPages"),
        "extraction_status": "success" if len(body.strip()) >= 100 else "short_or_empty",
    }


## 7. 본문 수집 실행

중단되더라도 이미 저장된 결과를 다시 요청하지 않도록 `articles_partial.csv`에
주기적으로 저장합니다.

처음에는 **API 테스트 비용을 아끼기 위해 `TEST_LIMIT`을 10~20 정도로 두고 실행**한 뒤,
결과가 정상임을 확인하고 `None`으로 바꾸어 전체 수집합니다.


In [ ]:
TEST_LIMIT = 20  # 테스트 후 전체 수집하려면 None

partial_file = DATA_DIR / "articles_partial.csv"
failed_file = DATA_DIR / "failed_urls_partial.csv"

if partial_file.exists():
    done_df = pd.read_csv(partial_file)
    done_ids = set(done_df["article_id"].astype(str))
else:
    done_df = pd.DataFrame()
    done_ids = set()

targets = article_index[~article_index["article_id"].astype(str).isin(done_ids)].copy()

if TEST_LIMIT is not None:
    targets = targets.head(TEST_LIMIT)

article_rows = []
failed_rows = []

for _, row in tqdm(targets.iterrows(), total=len(targets), desc="Diffbot Article"):
    try:
        obj, raw = diffbot_article(row["url"], include_discussion=True)

        article_rows.append(
            flatten_article(row["article_id"], row["url"], obj)
        )

    except Exception as e:
        failed_rows.append({
            "article_id": row["article_id"],
            "source_url": row["url"],
            "error": str(e),
        })

    # 너무 빠른 연속 호출을 피하기 위한 짧은 간격
    time.sleep(0.2)

new_articles_df = pd.DataFrame(article_rows)

if len(done_df):
    articles_df = pd.concat([done_df, new_articles_df], ignore_index=True)
else:
    articles_df = new_articles_df.copy()

articles_df = articles_df.drop_duplicates("article_id")
articles_df.to_csv(partial_file, index=False, encoding="utf-8-sig")

pd.DataFrame(failed_rows).to_csv(
    failed_file, index=False, encoding="utf-8-sig"
)

print("현재 Article 결과:", len(articles_df))
print("본문 100자 이상:", (articles_df["body_length"] >= 100).sum())
print("실패:", len(failed_rows))


## 8. 댓글을 별도 테이블로 추출

Article API 응답에 `discussion`이 포함될 수 있습니다.
댓글은 기사와 분리해서 저장합니다.

Diffbot Discussion 구조에서는 각 post에 `id`, `parentId`, `text`,
`date`, `author`, `humanLanguage` 등이 들어올 수 있습니다.


In [ ]:
def extract_comments_from_article_object(article_id, obj):
    rows = []

    if not obj:
        return rows

    discussion = obj.get("discussion")
    if not discussion:
        return rows

    posts = discussion.get("posts") or []

    for post in posts:
        text = post.get("text") or ""

        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": text,
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 9. 댓글을 확실하게 별도 수집하고 싶을 때

Article API의 `discussion`이 비어 있는 사이트는 Discussion API를 한 번 더 호출할 수 있습니다.
다만 이것은 **모든 기사에 무조건 두 번 요청하지 않도록** 설계하는 것이 좋습니다.

아래 함수는 필요할 때만 Discussion API를 호출합니다.


In [ ]:
def diffbot_discussion(url, timeout_ms=60000):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "maxPages": "all",
    }

    r = requests.get(DIFFBOT_DISCUSSION_URL, params=params, timeout=90)

    if r.status_code == 429:
        raise RuntimeError("Diffbot Discussion rate limit(429)입니다.")

    r.raise_for_status()
    data = r.json()

    objects = data.get("objects") or []
    return objects[0] if objects else None

def flatten_discussion(article_id, discussion_obj):
    rows = []

    if not discussion_obj:
        return rows

    for post in discussion_obj.get("posts") or []:
        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": post.get("text"),
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 10. 현재 Article 결과에서 댓글이 없는 기사만 Discussion API로 보완

이 단계는 API 비용이 추가될 수 있으므로 필요할 때 실행합니다.


In [ ]:
RUN_DISCUSSION_BACKFILL = False  # 비용/요청량을 확인한 후 True

comments_rows = []

if RUN_DISCUSSION_BACKFILL:
    for _, row in tqdm(
        articles_df[articles_df["source_url"].notna()].iterrows(),
        total=articles_df["source_url"].notna().sum(),
        desc="Diffbot Discussion"
    ):
        try:
            discussion = diffbot_discussion(row["source_url"])
            comments_rows.extend(
                flatten_discussion(row["article_id"], discussion)
            )
        except Exception as e:
            print("댓글 실패:", row["source_url"], e)

        time.sleep(0.2)

comments_df = pd.DataFrame(comments_rows)

if len(comments_df):
    comments_df = comments_df.drop_duplicates(
        subset=["article_id", "comment_id", "comment_text"]
    )
    comments_df.to_csv(COMMENTS_FILE, index=False, encoding="utf-8-sig")
    print("댓글 수:", len(comments_df))
else:
    print("댓글 데이터가 없습니다. RUN_DISCUSSION_BACKFILL=True로 실행하세요.")


## 11. 수집 결과 검증

기존 코드에서는 본문 결측 제거 후 약 3,976건만 실제 분석에 사용되었습니다.
이번에는 **수집된 기사 수 / 본문 성공 / 빈 본문 / 영어 / 국가 미상 / 댓글 수**를
명시적으로 확인합니다.


In [ ]:
# 전체 저장
articles_df.to_csv(ARTICLES_FILE, index=False, encoding="utf-8-sig")

if len(articles_df):
    print("===== ARTICLE COLLECTION REPORT =====")
    print("기사 수:", len(articles_df))
    print("본문 100자 이상:", (articles_df["body_length"] >= 100).sum())
    print("본문 100자 미만:", (articles_df["body_length"] < 100).sum())
    print("언어 en:", (articles_df["language"].astype(str).str.lower() == "en").sum())
    print("국가 미상:", articles_df["publisher_country"].isna().sum())
    print("언론사 미상:", articles_df["source"].isna().sum())

    display(
        articles_df[
            ["article_id", "title", "source", "publisher_country",
             "publisher_region", "language", "body_length", "extraction_status"]
        ].head(20)
    )


## 12. Excel 파일 생성

분석용으로는 CSV가 더 안정적이고,
확인/제출용으로는 Excel을 같이 만들어 둡니다.


In [ ]:
excel_path = DATA_DIR / "hormuz_2026_articles_comments.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    articles_df.to_excel(writer, sheet_name="articles", index=False)

    if COMMENTS_FILE.exists():
        comments_for_excel = pd.read_csv(COMMENTS_FILE)
        comments_for_excel.to_excel(writer, sheet_name="comments", index=False)

print("저장 완료:", excel_path)


# 13. 기존 텍스트마이닝 코드와 연결

여기부터는 기존 노트북의 NLTK 전처리 코드를 연결하면 됩니다.

중요한 점은 `body`를 분석 대상으로 사용하고,
`title`은 별도 분석 컬럼으로 보존하는 것입니다.

또한 `hormuz`, `strait`은 연구 질문에 따라 불용어로 제거할 수 있지만,
`iran`, `iranian`, `israel`, `israeli`, `us`, `america` 등을 무조건 제거하면
국가/행위자 프레임 분석에 필요한 정보가 사라질 수 있으므로 신중하게 결정합니다.


In [ ]:
# 텍스트마이닝 시작용 최소 코드

analysis_df = articles_df.copy()

# 실제 본문이 충분한 기사만 분석
analysis_df = analysis_df[
    (analysis_df["language"].astype(str).str.lower() == "en") &
    (analysis_df["body_length"] >= 100)
].copy()

analysis_df["text_for_mining"] = (
    analysis_df["title"].fillna("") + " " +
    analysis_df["body"].fillna("")
)

print("텍스트마이닝 대상 기사:", len(analysis_df))
analysis_df[[
    "title", "source", "publisher_country",
    "publisher_region", "published_at", "body_length"
]].head()


## 14. 중단 후 재시작

`articles_partial.csv`가 남아 있으면 이미 처리된 `article_id`는 다시 호출하지 않습니다.

따라서 전체 수집 중 Jupyter/VS Code가 종료되어도
처음부터 다시 시작할 필요가 없습니다.

권장 실행 순서:

1. `TEST_LIMIT = 20`
2. 20건 결과 확인
3. `TEST_LIMIT = None`
4. 전체 Article 수집
5. `RUN_DISCUSSION_BACKFILL = True`는 필요할 때 별도 실행
6. 최종 CSV/Excel 생성
